# Лабораторная работа №4 — параллельная обработка запросов

Ноутбук запускается в Google Colab. Никакие сторонние библиотеки и загрузка файлов не нужны.

Выполняйте ячейки сверху вниз. Чтобы изменить эксперимент, отредактируйте значения в первой ячейке с кодом.

> Время работы имитируется через `sleep`. Проценты CPU из задания показаны как параметры модели, но реальная загрузка процессора не измеряется.

In [1]:
import asyncio
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# Настройки эксперимента — эти значения можно менять.
USER_COUNTS = [3, 3, 3]       # Регистрация, главная страница, список пользователей
REQUEST_TIMES = [1.0, 0.7, 0.3]  # Время одного запроса в секундах

ACTIONS = [
    {"name": "Регистрация пользователя", "cpu": 25},
    {"name": "Главная страница", "cpu": 15},
    {"name": "Список пользователей", "cpu": 1},
]

if len(USER_COUNTS) != 3 or len(REQUEST_TIMES) != 3:
    raise ValueError("Нужно указать ровно три значения для каждого списка.")
if any(count < 0 for count in USER_COUNTS):
    raise ValueError("Количество пользователей не может быть отрицательным.")
if any(duration < 0 for duration in REQUEST_TIMES):
    raise ValueError("Время запроса не может быть отрицательным.")

requests = []
request_id = 1
for action, count, duration in zip(ACTIONS, USER_COUNTS, REQUEST_TIMES):
    for _ in range(count):
        requests.append({"id": request_id, "name": action["name"], "duration": duration})
        request_id += 1

print("Параметры эксперимента:")
for action, count, duration in zip(ACTIONS, USER_COUNTS, REQUEST_TIMES):
    print(f"{action['name']}: {count} пользователей, {duration:.2f} сек., условная нагрузка CPU — {action['cpu']}%")
print(f"Всего запросов: {len(requests)}")


Параметры эксперимента:
Регистрация пользователя: 3 пользователей, 1.00 сек., условная нагрузка CPU — 25%
Главная страница: 3 пользователей, 0.70 сек., условная нагрузка CPU — 15%
Список пользователей: 3 пользователей, 0.30 сек., условная нагрузка CPU — 1%
Всего запросов: 9


## 1. Последовательная обработка

Каждый запрос начинается после завершения предыдущего.

In [2]:
def run_sequential(request_list):
    for request in request_list:
        time.sleep(request["duration"])
        print(f"Запрос {request['id']}: {request['name']} — выполнен")

start = time.perf_counter()
run_sequential(requests)
sequential_time = time.perf_counter() - start
print(f"Время последовательной обработки: {sequential_time:.3f} сек.")


Запрос 1: Регистрация пользователя — выполнен
Запрос 2: Регистрация пользователя — выполнен
Запрос 3: Регистрация пользователя — выполнен
Запрос 4: Главная страница — выполнен
Запрос 5: Главная страница — выполнен
Запрос 6: Главная страница — выполнен
Запрос 7: Список пользователей — выполнен
Запрос 8: Список пользователей — выполнен
Запрос 9: Список пользователей — выполнен
Время последовательной обработки: 6.004 сек.


## 2. Параллельная обработка через subprocess

Запускаем отдельный дочерний процесс для каждого типа действий. Внутри каждого процесса запросы выполняются последовательно.

In [3]:
# Создаём небольшой файл-обработчик прямо в среде Colab.
worker_code = '''import sys
import time

action_name = sys.argv[1]
first_id = int(sys.argv[2])
count = int(sys.argv[3])
duration = float(sys.argv[4])

for request_id in range(first_id, first_id + count):
    time.sleep(duration)
    print(f"Запрос {request_id}: {action_name} — выполнен")
'''
worker_path = Path("/content/lab4_worker.py")
worker_path.write_text(worker_code, encoding="utf-8")

def run_subprocesses(request_list):
    processes = []
    for action in ACTIONS:
        group = [item for item in request_list if item["name"] == action["name"]]
        if not group:
            continue
        process = subprocess.Popen(
            [sys.executable, str(worker_path), action["name"], str(group[0]["id"]),
             str(len(group)), str(group[0]["duration"])],
            stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
        )
        processes.append(process)

    for process in processes:
        output, errors = process.communicate()
        if process.returncode != 0:
            raise RuntimeError(errors or "Ошибка дочернего процесса")
        print(output, end="")

start = time.perf_counter()
run_subprocesses(requests)
subprocess_time = time.perf_counter() - start
print(f"Время subprocess: {subprocess_time:.3f} сек.")


Запрос 1: Регистрация пользователя — выполнен
Запрос 2: Регистрация пользователя — выполнен
Запрос 3: Регистрация пользователя — выполнен
Запрос 4: Главная страница — выполнен
Запрос 5: Главная страница — выполнен
Запрос 6: Главная страница — выполнен
Запрос 7: Список пользователей — выполнен
Запрос 8: Список пользователей — выполнен
Запрос 9: Список пользователей — выполнен
Время subprocess: 3.071 сек.


## 3. Параллельная обработка через threads

Для выполнения запросов используем пул потоков `ThreadPoolExecutor`.

In [4]:
def run_one_request(request):
    time.sleep(request["duration"])
    return f"Запрос {request['id']}: {request['name']} — выполнен"

def run_threads(request_list):
    with ThreadPoolExecutor(max_workers=min(32, len(request_list))) as pool:
        for result in pool.map(run_one_request, request_list):
            print(result)

if requests:
    start = time.perf_counter()
    run_threads(requests)
    threads_time = time.perf_counter() - start
else:
    threads_time = 0.0
print(f"Время threads: {threads_time:.3f} сек.")


Запрос 1: Регистрация пользователя — выполнен
Запрос 2: Регистрация пользователя — выполнен
Запрос 3: Регистрация пользователя — выполнен
Запрос 4: Главная страница — выполнен
Запрос 5: Главная страница — выполнен
Запрос 6: Главная страница — выполнен
Запрос 7: Список пользователей — выполнен
Запрос 8: Список пользователей — выполнен
Запрос 9: Список пользователей — выполнен
Время threads: 1.003 сек.


## 4. Асинхронная обработка через asyncio

`asyncio.gather` запускает асинхронные задачи совместно. Во время `asyncio.sleep` цикл событий может переключаться на другие задачи.

In [5]:
async def run_one_async(request):
    await asyncio.sleep(request["duration"])
    return f"Запрос {request['id']}: {request['name']} — выполнен"

async def run_async(request_list):
    results = await asyncio.gather(*(run_one_async(item) for item in request_list))
    for result in results:
        print(result)

start = time.perf_counter()
await run_async(requests)  # В Colab можно использовать await прямо в ячейке.
asyncio_time = time.perf_counter() - start
print(f"Время asyncio: {asyncio_time:.3f} сек.")


Запрос 1: Регистрация пользователя — выполнен
Запрос 2: Регистрация пользователя — выполнен
Запрос 3: Регистрация пользователя — выполнен
Запрос 4: Главная страница — выполнен
Запрос 5: Главная страница — выполнен
Запрос 6: Главная страница — выполнен
Запрос 7: Список пользователей — выполнен
Запрос 8: Список пользователей — выполнен
Запрос 9: Список пользователей — выполнен
Время asyncio: 1.002 сек.


## Сравнение результатов

Значения времени будут немного отличаться при каждом запуске и зависят от загруженности среды Colab.

In [6]:
print(f"{'Способ':<22} Время, сек.")
print("-" * 36)
for name, elapsed in [
    ("Последовательно", sequential_time),
    ("subprocess", subprocess_time),
    ("threads", threads_time),
    ("asyncio", asyncio_time),
]:
    print(f"{name:<22} {elapsed:.3f}")

print("\\nВывод: threads и asyncio позволяют одновременно ожидать несколько запросов.")
print("subprocess запускает отдельные процессы, поэтому имеет дополнительные затраты на запуск.")


Способ                 Время, сек.
------------------------------------
Последовательно        6.004
subprocess             3.071
threads                1.003
asyncio                1.002
\nВывод: threads и asyncio позволяют одновременно ожидать несколько запросов.
subprocess запускает отдельные процессы, поэтому имеет дополнительные затраты на запуск.
